# 🎙️ Voice Studio v2.6 — Director Mode (Colab Edition)

**Script do → AI voiceover MP3 lo** — free, Chatterbox engine ke saath.

### Chalane ka tareeqa
1. **Runtime > Run all** dabao (ya cells ek-ek karke Shift+Enter se chalao)
2. Agar GPU ka error aaye: **Runtime > Change runtime type > GPU (T4)** select karke dobara chalao
3. Step 3 wala cell chalne ke baad neeche tool khul jayega
4. Script paste karo → chaaho to voice sample upload karo (clone) → **Generate Voiceover**
5. MP3 **usi session me download kar lena** — Colab band hote hi files ur jati hain

> Tip: Apni LLM-annotated script (`[pause]`, `[short pause]`, `[long pause]`, `[slowly]`, `[sadly]`, `[emphatic]`, `[with growing tension]`...) seedha paste karo — tags khud handle ho jayenge, bole nahi jayenge. Default waqfa 'Waqfa (Pause)' slider se badhao/ghatao.

In [ ]:
# Step 0: Colab fix — torchvision ka version torch se mismatch karta hai aur model load rok deta hai.
# Voice tool ko torchvision ki zaroorat hi nahi, is liye ise hata do.
!pip uninstall -y torchvision
print("Fix applied ✅")

In [ ]:
# Step 1: Zaroori libraries install karo (pehli dafa 3-5 minute lag sakte hain)
!pip install -q chatterbox-tts gradio
print("Install done ✅")

In [ ]:
# Step 2: GPU check + model load (pehli dafa model download hoga, ~1.5 GB)
import torch

if not torch.cuda.is_available():
    raise SystemExit("❌ GPU nahi mila! Menu me jao: Runtime > Change runtime type > Hardware accelerator = GPU (T4) select karo, phir dobara chalao.")

print("GPU:", torch.cuda.get_device_name(0))

from chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")
SR = model.sr
print(f"Model ready ✅ (sample rate {SR})")

In [ ]:
# Step 3: Voice Studio app — ye cell chalao, neeche tool khul jayega
import re
import difflib
import subprocess
import numpy as np
import torch
import torchaudio as ta
import librosa
import gradio as gr

VERSION = "v2.6"

# ---------- Director Mode: annotated script seedha paste karo ----------
EMOTION_PRESETS = {
    'sad':     {'exaggeration': 0.35, 'cfg': 0.30, 'vol': 0.80, 'pitch': -3, 'rate': 0.88},
    'happy':   {'exaggeration': 0.75, 'cfg': 0.50, 'vol': 1.15, 'pitch': +3, 'rate': 1.05},
    'excited': {'exaggeration': 0.85, 'cfg': 0.55, 'vol': 1.25, 'pitch': +2, 'rate': 1.08},
    'angry':   {'exaggeration': 0.90, 'cfg': 0.40, 'vol': 1.35, 'pitch': -2, 'rate': 1.05},
    'soft':    {'exaggeration': 0.40, 'cfg': 0.40, 'vol': 0.65, 'pitch': 0,   'rate': 0.95},
    'loud':    {'exaggeration': 0.70, 'cfg': 0.50, 'vol': 1.50, 'pitch': 0,   'rate': 1.00},
    'slow':    {'exaggeration': 0.50, 'cfg': 0.25, 'vol': 1.00, 'pitch': -1,  'rate': 0.82},
    'fast':    {'exaggeration': 0.60, 'cfg': 0.70, 'vol': 1.05, 'pitch': +1,  'rate': 1.18},
    'high':    {'exaggeration': 0.60, 'cfg': 0.50, 'vol': 1.00, 'pitch': +4,  'rate': 1.00},
    'low':     {'exaggeration': 0.50, 'cfg': 0.40, 'vol': 1.05, 'pitch': -4,  'rate': 0.95},
    'tense':   {'exaggeration': 0.70, 'cfg': 0.35, 'vol': 1.10, 'pitch': -2,  'rate': 0.95},
}
TAG_WORDS = {
    'sad':'sad','sorrow':'sad','sorrowful':'sad','gloomy':'sad','depressed':'sad',
    'miserable':'sad','heartbroken':'sad','tearful':'sad','crying':'sad','cry':'sad',
    'weep':'sad','weeping':'sad','mournful':'sad','blue':'sad','downcast':'sad',
    'melancholy':'sad','grief':'sad','grieving':'sad','sobbing':'sad','sadly':'sad',
    'regretful':'sad',
    'happy':'happy','joyful':'happy','cheerful':'happy','glad':'happy',
    'delighted':'happy','elated':'happy','jubilant':'happy','merry':'happy',
    'joy':'happy','pleased':'happy','warmly':'happy','warm':'happy',
    'hopeful':'happy','hope':'happy','relieved':'happy','relief':'happy',
    'excited':'excited','enthusiastic':'excited','energetic':'excited',
    'eager':'excited','thrilled':'excited','pumped':'excited','surprised':'excited',
    'shocked':'excited','amazed':'excited','astonished':'excited','wonder':'excited',
    'dramatically':'excited','dramatic':'excited',
    'angry':'angry','furious':'angry','mad':'angry','rage':'angry','raging':'angry',
    'irritated':'angry','annoyed':'angry','harsh':'angry','stern':'angry',
    'fierce':'angry','intense':'angry','intensely':'angry','intensity':'angry',
    'with restrained anger':'angry','restrained anger':'angry',
    'voice hardens':'angry','hardens':'angry',
    'soft':'soft','whisper':'soft','whispered':'soft','whispering':'soft',
    'quiet':'soft','quietly':'soft','gentle':'soft','gently':'soft',
    'tender':'soft','tenderly':'soft','calm':'soft','calmly':'soft',
    'soothing':'soft','murmur':'soft','murmured':'soft','hushed':'soft',
    'whispery':'soft','romantic':'soft','loving':'soft','scared':'soft',
    'afraid':'soft','fearful':'soft','softly':'soft','almost whispering':'soft',
    'sympathetic':'soft','sympathy':'soft','voice softens':'soft','softens':'soft',
    'with quiet admiration':'soft','quiet admiration':'soft',
    'loud':'loud','loudly':'loud','shout':'loud','shouting':'loud','shouted':'loud',
    'scream':'loud','screaming':'loud','screamed':'loud','yell':'loud',
    'yelling':'loud','yelled':'loud','roar':'loud','roaring':'loud',
    'thunderous':'loud','confident':'loud','brave':'loud','proud':'loud',
    'with authority':'loud','authoritative':'loud','authority':'loud',
    'slow':'slow','slowly':'slow','sluggish':'slow','drawl':'slow',
    'drawling':'slow','leisurely':'slow','very slowly':'slow','reflective':'slow',
    'thoughtful':'slow','with emotional weight':'slow','emotional weight':'slow',
    'fast':'fast','quick':'fast','quickly':'fast','rapid':'fast','rapidly':'fast',
    'hurried':'fast','rushed':'fast','hasty':'fast','brisk':'fast','nervous':'fast',
    'faster':'fast','picking up pace':'fast','building momentum':'fast',
    'urgent':'fast','urgently':'fast','with urgency':'fast',
    'high':'high','squeaky':'high','shrill':'high','falsetto':'high',
    'confused':'high','confusion':'high','disbelieving':'high','disbelief':'high',
    'in disbelief':'high','with disbelief':'high','with curiosity':'high',
    'curious':'high','curiosity':'high','raising voice':'high','higher voice':'high',
    'high voice':'high',
    'low':'low','deep':'low','bass':'low','grave':'low','serious':'low',
    'seriously':'low','somber':'low','somberly':'low','solemn':'low',
    'solemnly':'low','firm':'low','firmly':'low','disturbed':'low','haunting':'low',
    'ominous':'low','darkly':'low','coldly':'low','grimly':'low','grim':'low',
    'dark':'low','menacing':'low','evil':'low','mysterious':'low',
    'lowering voice':'low','lower voice':'low','deep voice':'low',
    'tense':'tense','tension':'tense','uneasy':'tense','suspense':'tense',
    'suspenseful':'tense','with growing tension':'tense','building tension':'tense',
    'with growing suspense':'tense','building suspense':'tense',
    'raising intensity':'tense','building intensity':'tense',
    'with controlled intensity':'tense','controlled intensity':'tense','anxious':'tense',
}
PAUSE_TAGS = {'pause': 1.0, 'short pause': 0.5, 'long pause': 2.0,
              'pause briefly': 0.4, 'sigh': 0.6, 'sighs': 0.6,
              'breathes in': 0.6, 'breathing in': 0.6,
              'deep breath': 0.6, 'takes a breath': 0.6}
EMPH_TAGS = {'emphatic', 'emphatically'}
TAG_TOKEN_RE = re.compile(r"(\[[^\]]*\])")
TAG_PARSE_RE = re.compile(r"\[([a-z][a-z ]*)(?::\s*([\d.]+))?\]", re.IGNORECASE)
SAD_WORDS = {'sad','tears','cry','cried','died','death','lost','lonely','miss','missed',
             'grief','sorrow','painful','heartbreak','alone','goodbye','afraid','fear'}
HAPPY_WORDS = {'happy','joy','laugh','celebrat','wonderful','amazing','excit','love',
               'delight','beautiful','proud'}
FEAR_WORDS = {'afraid','fear','feared','terror','terrible','dread','dreadful','panic',
              'panicked','horror','horrifying','nightmare','scared','frightening','creepy'}
ANGER_WORDS = {'angry','anger','furious','fury','rage','raging','hate','hated','hatred',
               'wrath','outrage','outraged'}
SURPRISE_WORDS = {'shock','shocked','shocking','stunned','stunning','unbelievable',
                  'incredible','astonishing','astonished','amazed','amazement',
                  'miracle','miraculous'}
CONTRAST_STARTERS = ('but ', 'however,', 'except', 'instead', 'yet ', 'although ',
                     'though ', 'nevertheless,', 'the problem was', 'what they didn',
                     'the strange part', 'it wasn', 'it was not', 'the truth was',
                     'that should have been the end')
NUMBER_WORDS = {'hundred','thousand','million','billion','dozen','percent',
                'twenty','thirty','forty','fifty','sixty','seventy','eighty','ninety'}

def resolve_emo(name):
    if name in EMOTION_PRESETS:
        return name
    if name in TAG_WORDS:
        return TAG_WORDS[name]
    close = difflib.get_close_matches(name, TAG_WORDS.keys(), n=1, cutoff=0.75)
    return TAG_WORDS[close[0]] if close else None

def _wb(words):
    return re.compile(r"\b(?:" + "|".join(words) + r")\b")

SAD_RE = _wb(SAD_WORDS)
HAPPY_RE = _wb(HAPPY_WORDS)
FEAR_RE = _wb(FEAR_WORDS)
ANGER_RE = _wb(ANGER_WORDS)
SURPRISE_RE = _wb(SURPRISE_WORDS)
NUMBER_RE = _wb(NUMBER_WORDS)

def auto_direct(text):
    """Bina tags ke script khud direct karo. Returns (emo, emphasis, pause_after, starts_contrast)."""
    t = text.strip()
    tl = t.lower()
    emo, emphasis, pause_after = None, False, None
    if SAD_RE.search(tl):
        emo = 'sad'
    elif FEAR_RE.search(tl):
        emo = 'tense'
    elif ANGER_RE.search(tl):
        emo = 'angry'
    elif HAPPY_RE.search(tl):
        emo = 'happy'
    elif SURPRISE_RE.search(tl):
        emo = 'excited'
    if t.endswith('!') and not emo:
        emo = 'excited'
    elif t.endswith('?') and not emo:
        emo = 'high'
    if len(t) < 40 and t.endswith('.') and not emo:
        emo = 'slow'       # chhota payoff jumla -> aahista, zor se, waqfe ke saath
        emphasis = True
        pause_after = 0.6
    if not emo and (re.search(r"\d", t) or NUMBER_RE.search(tl)):
        emo = 'slow'       # numbers/statistics -> aahista
    starts_contrast = tl.startswith(CONTRAST_STARTERS)
    return emo, emphasis, pause_after, starts_contrast

def split_sentences(text, max_chars=200):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, cur = [], ""
    for s in sentences:
        if len(cur) + len(s) + 1 <= max_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            while len(s) > max_chars:
                chunks.append(s[:max_chars])
                s = s[max_chars:]
            cur = s
    if cur:
        chunks.append(cur)
    return [c for c in chunks if c]

def trim_edges(y, sr, thresh=0.015):
    """Har chunk ke shuru/end ki extra khamoshi kaato — taake sirf slider wala waqfa rahe."""
    idx = np.where(np.abs(y) > thresh)[0]
    if len(idx) == 0:
        return y
    start = max(0, idx[0] - int(sr * 0.03))
    end = min(len(y), idx[-1] + int(sr * 0.05))
    return y[start:end]

def parse_units(script, max_chars, unknown):
    """Annotated script -> units: har unit {text, emo, emphasis, pause_after}."""
    text = re.sub(r"\s+", " ", script).strip()
    if not text:
        return []
    units = []
    emo = None        # persistent emotion (naya tag aane tak)
    emph_next = False
    for tok in TAG_TOKEN_RE.split(text):
        if not tok.strip():
            continue
        pm = TAG_PARSE_RE.fullmatch(tok.strip())
        if not pm and tok.strip().startswith('['):
            unknown.append(tok.strip())  # samajh na aaya tag -> strip, kabhi bola nahi jayega
            continue
        if pm:
            name = pm.group(1).lower().strip()
            val = pm.group(2)
            if name == 'pause' and val:
                if units:
                    units[-1]['pause_after'] = float(val)
                continue
            if name in PAUSE_TAGS:
                if units:
                    units[-1]['pause_after'] = PAUSE_TAGS[name]
                continue
            if name in EMPH_TAGS:
                if units and units[-1]['pause_after'] is None:
                    units[-1]['pause_after'] = 0.05  # tootay baghair jor
                emph_next = True
                continue
            r = resolve_emo(name)
            if r:
                emo = r
            else:
                unknown.append(name)
            continue
        for s in split_sentences(tok, max_chars):
            units.append({'text': s, 'emo': emo, 'emphasis': emph_next,
                          'pause_after': None})
            emph_next = False
    return [u for u in units if u['text']]

def generate_voiceover(script, ref_audio, exaggeration, cfg_weight, pause_sec,
                       auto_emotion, progress=gr.Progress()):
    try:
        unknown = []
        units = parse_units(script, 200, unknown)
        if not units:
            return None, None, "❌ Script khali hai — pehle text paste karo."
        prompt = ref_audio  # clone ke liye sample file, warna None (default voice)
        wavs = []
        emo_count = {}
        conds_ready = False
        for u in progress.tqdm(units, desc="Voice ban rahi hai..."):
            if u['emo']:
                emo_name, emphasis, pause_ov = u['emo'], u['emphasis'], u['pause_after']
            elif auto_emotion:
                emo_name, emphasis, pause_ov, starts_contrast = auto_direct(u['text'])
                emphasis = emphasis or u['emphasis']
                if pause_ov is None:
                    pause_ov = u['pause_after']
                if starts_contrast and len(wavs) >= 2:
                    wavs[-1] = torch.cat([wavs[-1], torch.zeros(int(SR * 0.8))])
            else:
                emo_name, emphasis, pause_ov = None, u['emphasis'], u['pause_after']
            if emo_name in EMOTION_PRESETS:
                p = EMOTION_PRESETS[emo_name]
                ex, cw, vol, pitch = p['exaggeration'], p['cfg'], p['vol'], p['pitch']
                rate = p.get('rate', 1.0)
                emo_count[emo_name] = emo_count.get(emo_name, 0) + 1
            else:
                ex, cw, vol, pitch, rate = exaggeration, cfg_weight, 1.0, 0, 1.0
            if emphasis:
                vol *= 1.25
                ex = min(1.0, ex + 0.15)
            ap = prompt if (prompt and not conds_ready) else None
            wav = model.generate(u['text'], audio_prompt_path=ap,
                                 exaggeration=ex, cfg_weight=cw)
            conds_ready = True
            wav = wav.squeeze(0) if wav.dim() > 1 else wav
            y = wav.cpu().numpy().astype(np.float32)
            if pitch:
                try:
                    y = librosa.effects.pitch_shift(y, sr=SR, n_steps=pitch).astype(np.float32)
                except Exception:
                    pass
            if rate != 1.0:
                try:
                    y = librosa.effects.time_stretch(y, rate=rate).astype(np.float32)
                except Exception:
                    pass
            if vol != 1.0:
                y = y * vol
            y = np.clip(y, -1.0, 1.0)
            y = trim_edges(y, SR)  # model ki apni extra khamoshi hatao
            wavs.append(torch.from_numpy(y))
            gap = pause_ov if pause_ov is not None else pause_sec
            wavs.append(torch.zeros(int(SR * gap)))
        if not wavs:
            return None, None, "❌ Koi sentence nahi mila."
        full = torch.cat(wavs[:-1]).unsqueeze(0)  # aakhri silence hata do
        ta.save("/tmp/voiceover.wav", full, SR)
        subprocess.run(
            ["ffmpeg", "-y", "-i", "/tmp/voiceover.wav",
             "-codec:a", "libmp3lame", "-b:a", "192k", "/tmp/voiceover.mp3"],
            check=True, capture_output=True,
        )
        n_min = int(full.shape[1] / SR / 60)
        n_sec = int(full.shape[1] / SR % 60)
        emo_txt = ", ".join(f"{k}:{v}" for k, v in sorted(emo_count.items()))
        msg = f"Done ✅ {len(units)} parts, ~{n_min}:{n_sec:02d}"
        if emo_txt:
            msg += f" | moods: {emo_txt}"
        unk = sorted(set(unknown))
        if unk:
            msg += f" | na-samjhe tags (ignore hue): {', '.join(unk)}"
        msg += " — MP3 download kar lo."
        print(msg)
        return "/tmp/voiceover.mp3", "/tmp/voiceover.mp3", msg
    except Exception as e:
        import traceback
        traceback.print_exc()
        return None, None, f"❌ Error ({type(e).__name__}): {e}"

with gr.Blocks(title="Voice Studio") as demo:
    gr.Markdown(f"# 🎙️ Voice Studio {VERSION} — Script se Voiceover")
    gr.Markdown("Script paste karo, chaaho to apni voice ki 5–10 second ki sample do (clone), Generate dabao, MP3 download kar lo.")
    gr.Markdown("🎭 **Annotated script seedha paste karo** — `[pause]` `[short pause]` `[long pause]` `[slowly]` `[very slowly]` `[quietly]` `[softly]` `[whispering]` `[serious]` `[somber]` `[emphatic]` `[firmly]` `[intense]` `[tense]` `[urgent]` `[calmly]` `[reflective]` `[surprised]` `[shocked]` `[uneasy]` `[ominous]` `[darkly]` `[coldly]` `[grimly]` `[sadly]` `[heartbroken]` `[warmly]` `[hopeful]` `[excited]` `[energetic]` `[sighs]` `[breathes in]` `[lowering voice]` `[raising intensity]` `[voice softens]` `[with growing tension]` `[with quiet admiration]`... **koi bhi direction word chalega, tags bole nahi jayenge.** Tags lagana zaroori nahi — saadi script paste karo to Auto Director khud faisla karega: sawal unchi pitch me, hairat energetic, gham/khushi/gussa/dar ke lafz apne mood me, chhote payoff jumle aahista zor se waqfe ke saath, numbers aahista, aur 'but/however' wale mor se pehle waqfa.")
    script = gr.Textbox(label="Script (English)", lines=12,
                        placeholder="Apni poori script yahan paste karo... tags ke saath bhi chalegi.")
    ref = gr.Audio(label="Voice sample — clone ke liye (optional, 5-10 sec ki saaf recording)",
                   type="filepath")
    with gr.Row():
        exagg = gr.Slider(0.25, 1.0, value=0.5, step=0.05,
                          label="Emotion (Exaggeration) — default mood")
        cfg = gr.Slider(0.0, 1.0, value=0.5, step=0.05,
                        label="Pacing (Cfg weight) — default raftaar")
    with gr.Row():
        pause = gr.Slider(0.1, 1.5, value=0.25, step=0.05,
                          label="Waqfa (Pause) — default, har sentence ke baad kitne second")
        auto_emo = gr.Checkbox(label="Auto emotion — bina tag wale sentences ka mood khud andaza lagao", value=True)
    btn = gr.Button("🎧 Generate Voiceover", variant="primary")
    status = gr.Textbox(label="Status", interactive=False,
                        placeholder="Yahan progress / error nazar aayega...")
    out_audio = gr.Audio(label="Sun lo")
    out_file = gr.File(label="MP3 download")
    btn.click(generate_voiceover,
              inputs=[script, ref, exagg, cfg, pause, auto_emo],
              outputs=[out_audio, out_file, status])

demo.launch()
print("App band ho gayi. Dobara chalane ke liye ye cell phir se chalao.")
